### Download Survival Data


In [19]:
import requests
import pandas as pd
from io import StringIO

# On définit les URLs pour les types de cancer (exemples de Progenetix)
sampletable_urls = {
    "TCGA_ESCA": "https://progenetix.org/services/sampletable/?filters=pgx:TCGA-ESCA",

}

In [20]:
# Download sample tables and store them in a dictionary.

sampletables = {}

for source_query, url in sampletable_urls.items():
    print(f"Downloading sample table for {source_query}")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    df_tmp = pd.read_csv(StringIO(response.text), sep="\t")
    df_tmp["source_query"] = source_query
    sampletables[source_query] = df_tmp

In [21]:
#if many tables, concatenate them into a single DataFrame
all_samples = pd.concat(sampletables.values(), ignore_index=True)

In [22]:
#name of the columns
print(all_samples.columns)

# Clean table with just the column needed for survival analysis
survival_Ondata = all_samples[['biosample_id', 'followup_days', 'biosample_status_id']].copy()

#  Drop the data withotut followup days, as they cannot be used for survival analysis
survival_data = survival_Ondata.dropna(subset=['followup_days'])

Index(['biosample_id', 'individual_id', 'biosample_name', 'notes',
       'histological_diagnosis_id', 'histological_diagnosis_label',
       'pathological_stage_id', 'pathological_stage_label',
       'biosample_status_id', 'biosample_status_label',
       'sample_origin_type_id', 'sample_origin_type_label',
       'sampled_tissue_id', 'sampled_tissue_label', 'tnm', 'tumor_grade_id',
       'tumor_grade_label', 'age_iso', 'age_days', 'icdo_morphology_id',
       'icdo_morphology_label', 'icdo_topography_id', 'icdo_topography_label',
       'pubmed_id', 'pubmed_label', 'cellosaurus_id', 'cellosaurus_label',
       'cbioportal_id', 'cbioportal_label', 'tcgaproject_id',
       'tcgaproject_label', 'cohorts', 'geoprov_id', 'geoprov_city',
       'geoprov_country', 'geoprov_iso_alpha3', 'geoprov_long_lat',
       'followup_days', 'sex_id', 'sex_label', 'group_id', 'group_label',
       'source_query'],
      dtype='str')


In [ ]:
# i"m using biosample_status_id normally it has 'EFO:0009654', 'EFO:0009656' but not sure if it's survival or death
# Dictionnary for the mapping
#  0 = Vivant (Censured), 1 = Décédé (Event)
status_map = {
    'EFO:0009654': 0, # Alive
    'EFO:0009656': 1  # Dead
}

# transformation
survival_data['status'] = survival_data['biosample_status_id'].map(status_map)

# Verification of the transformation, looks if there are any remaining empty values
survival_data = survival_data.dropna(subset=['status'])


print(survival_data.head())

Prêt ! Nous avons 0 échantillons avec temps et statut.
Empty DataFrame
Columns: [biosample_id, followup_days, biosample_status_id, status]
Index: []
